# Módulo 2 — Sentimiento: BETO Fine-tuning
**dccuchile/bert-base-spanish-wwm-uncased** con Hugging Face Trainer

> Activar GPU: Runtime → Change runtime type → T4 GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/smartretail360'
MODEL_DIR  = f'{DRIVE_ROOT}/models/sentiment_analyzer/beto_finetuned'

import os
os.makedirs(MODEL_DIR, exist_ok=True)

In [ ]:
!pip install -q transformers datasets torch scikit-learn

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

MODEL_NAME = 'dccuchile/bert-base-spanish-wwm-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# amazon_reviews_multi usa scripts incompatibles con datasets >= 4.0.
# mteb/amazon_reviews_multi: mismo corpus, parquet nativo.
# Adaptamos esquema (text→review_body, label 0-4→stars 1-5) para que el resto no cambie.
ds = load_dataset('mteb/amazon_reviews_multi', 'es')
ds = ds.rename_columns({'text': 'review_body'}).map(
    lambda b: {'stars': [l + 1 for l in b['label']]},
    batched=True, remove_columns=['label', 'id']
)

def rating_to_label(r):
    return 0 if r <= 2 else (1 if r == 3 else 2)

def tokenize_and_label(batch):
    out = tokenizer(batch['review_body'], truncation=True, max_length=256)
    out['labels'] = [rating_to_label(s) for s in batch['stars']]
    return out

ds = ds.map(tokenize_and_label, batched=True, remove_columns=ds['train'].column_names)
print(ds)

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from sklearn.metrics import f1_score
import numpy as np

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {'f1_macro': f1_score(labels, preds, average='macro')}

args = TrainingArguments(
    output_dir=MODEL_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    seed=42,
    logging_steps=200,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds['train'],
    eval_dataset=ds['validation'],
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)
print(f'BETO guardado en {MODEL_DIR}')